In [0]:
from pyspark.sql import functions as F
from datetime import datetime

In [0]:
CATALOG = "clinical_trial_supply_chain"
BRONZE_SCHEMA = "bronze"

LANDING_ROOT = "/Workspace/Clinical_Trial_Supply_Chain/landing"
REFERENCE_ROOT = "/Workspace/Clinical_Trial_Supply_Chain/reference_data"

LOAD_ID = f"LOAD_{datetime.now().strftime('%Y%m%d_%H%M%S')}"

TABLES = {
    "site_master": f"{CATALOG}.{BRONZE_SCHEMA}.bronze_site_master",
    "shipment_events": f"{CATALOG}.{BRONZE_SCHEMA}.bronze_shipment_events",
    "inventory": f"{CATALOG}.{BRONZE_SCHEMA}.bronze_inventory_snapshot",
    "enrollment": f"{CATALOG}.{BRONZE_SCHEMA}.bronze_enrollment",
    "dispensing": f"{CATALOG}.{BRONZE_SCHEMA}.bronze_dispensing",
    "drug_master": f"{CATALOG}.{BRONZE_SCHEMA}.bronze_drug_master",
    "country_master": f"{CATALOG}.{BRONZE_SCHEMA}.bronze_country_master",
    "depot_master": f"{CATALOG}.{BRONZE_SCHEMA}.bronze_depot_master"
}

PROCESSED_TABLE = f"{CATALOG}.{BRONZE_SCHEMA}.bronze_processed_files"
AUDIT_TABLE = f"{CATALOG}.{BRONZE_SCHEMA}.bronze_load_audit"

In [0]:
def get_processed_files():

    try:

        rows = (
            spark.table(PROCESSED_TABLE)
                 .select("source_system", "file_name")
                 .collect()
        )

        return {(row["source_system"], row["file_name"]) for row in rows}

    except Exception as e:

        print(f"Error reading processed table: {e}")
        return set()


PROCESSED_FILES = get_processed_files()

print(f"Previously processed files : {len(PROCESSED_FILES)}")

In [0]:
def write_audit(
    table_name,
    file_name,
    records_loaded,
    start_time,
    end_time,
    status
):

    audit_df = spark.createDataFrame(
        [
            (
                LOAD_ID,
                table_name,
                file_name,
                records_loaded,
                start_time,
                end_time,
                status
            )
        ],
        [
            "load_id",
            "table_name",
            "source_file_name",
            "records_loaded",
            "load_start_timestamp",
            "load_end_timestamp",
            "load_status"
        ]
    )

    audit_df.write.mode("append").saveAsTable(AUDIT_TABLE)

In [0]:
def mark_processed(
    file_name,
    source_system,
    file_size
):

    processed_df = spark.createDataFrame(
        [
            (
                file_name,
                source_system,
                file_size,
                LOAD_ID,
                datetime.now()
            )
        ],
        [
            "file_name",
            "source_system",
            "file_size_bytes",
            "load_id",
            "processed_timestamp"
        ]
    )

    processed_df.write.mode("append").saveAsTable(PROCESSED_TABLE)

In [0]:
def read_file(file_path, file_format):

    if file_format == "csv":

        return (
            spark.read
                 .option("header", "true")
                 .csv(file_path)
        )

    elif file_format == "json":

        return (
            spark.read
                .option("multiline", "true")
                .json(file_path)
        )
    elif file_format == "parquet":

        return spark.read.parquet(file_path)

    elif file_format == "xml":

        return (
            spark.read
                 .format("xml")
                 .option("rowTag", "dispense")
                 .load(file_path)
        )

    else:
        raise Exception(f"Unsupported format : {file_format}")

In [0]:
def process_source(
    source_system,
    source_folder,
    bronze_table,
    file_format
):

    files = dbutils.fs.ls(source_folder)

    for file in files:

        file_key = (source_system, file.name)

        if file_key in PROCESSED_FILES:
            print(f"Skipping : {file.name}")
            continue

        start_time = datetime.now()

        try:

            print(f"Processing : {file.name}")

            df = read_file(file.path, file_format)

            df = (
                df.withColumn(
                    "source_file_name",
                    F.lit(file.name)
                )
                .withColumn(
                    "ingestion_timestamp",
                    F.current_timestamp()
                )
                .withColumn(
                    "load_id",
                    F.lit(LOAD_ID)
                )
            )

            record_count = df.count()

            (
                df.write
                  .format("delta")
                  .mode("append")
                  .saveAsTable(bronze_table)
            )

            mark_processed(
                file.name,
                source_system,
                file.size
            )

            write_audit(
                bronze_table,
                file.name,
                record_count,
                start_time,
                datetime.now(),
                "SUCCESS"
            )

            print(f"Loaded {record_count} records")

        except Exception as e:

            write_audit(
                bronze_table,
                file.name,
                0,
                start_time,
                datetime.now(),
                f"FAILED : {str(e)}"
            )

            print(f"FAILED : {file.name}")
            print(str(e))

In [0]:
process_source(
    source_system="site_master",
    source_folder=f"{LANDING_ROOT}/site_master",
    bronze_table=TABLES["site_master"],
    file_format="csv"
)

In [0]:
process_source(
    source_system="shipment_events",
    source_folder=f"{LANDING_ROOT}/shipment_events",
    bronze_table=TABLES["shipment_events"],
    file_format="json"
)

In [0]:
process_source(
    source_system="inventory",
    source_folder=f"{LANDING_ROOT}/inventory",
    bronze_table=TABLES["inventory"],
    file_format="csv"
)

In [0]:
process_source(
    source_system="enrollment",
    source_folder=f"{LANDING_ROOT}/enrollment",
    bronze_table=TABLES["enrollment"],
    file_format="parquet"
)

In [0]:
process_source(
    source_system="dispensing",
    source_folder=f"{LANDING_ROOT}/dispensing",
    bronze_table=TABLES["dispensing"],
    file_format="xml"
)

In [0]:
process_source(
    "drug_master",
    f"{REFERENCE_ROOT}/drug_master",
    TABLES["drug_master"],
    "csv"
)

process_source(
    "country_master",
    f"{REFERENCE_ROOT}/country_master",
    TABLES["country_master"],
    "csv"
)

process_source(
    "depot_master",
    f"{REFERENCE_ROOT}/depot_master",
    TABLES["depot_master"],
    "csv"
)

In [0]:
print("=" * 60)
print("BRONZE INGESTION COMPLETED")
print(f"LOAD ID : {LOAD_ID}")
print("=" * 60)

display(
    spark.sql(
        f"""
        SELECT *
        FROM {AUDIT_TABLE}
        WHERE load_id = '{LOAD_ID}'
        """
    )
)

In [0]:
%sql
SELECT *
FROM clinical_trial_supply_chain.bronze.bronze_processed_files;


In [0]:
%sql
SELECT
    source_system,
    file_name,
    COUNT(*) cnt
FROM clinical_trial_supply_chain.bronze.bronze_processed_files
GROUP BY source_system, file_name
HAVING COUNT(*) > 0